# Generando Texto con un Char-RNN

### Un modelo de lenguaje, letra por letra

Una Recurrent Neural Network (RNN) mantiene una memoria que se actualiza paso a paso. Aqui vamos a usar esa idea para algo concreto: un modelo que lee texto caracter por caracter, y aprende a predecir cual es el siguiente caracter mas probable.

 Este tipo de modelo se llama un **char-RNN**, y es uno de los primeros ejemplos historicos de lo que hoy llamamos un *modelo de lenguaje*.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## 1. Descargando el texto

Vamos a usar un subconjunto de las obras de Shakespeare (texto plano en ingles), publicado por Andrej Karpathy como parte de su proyecto `char-rnn`, uno de los proyectos que popularizo este tipo de modelo.

In [2]:
from pathlib import Path
import urllib.request


def descargar_texto_shakespeare():
    ruta = Path("datasets/shakespeare/shakespeare.txt")
    if not ruta.is_file():
        ruta.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, ruta)
    return ruta.read_text()


texto_shakespeare = descargar_texto_shakespeare()
print("Cantidad total de caracteres:", len(texto_shakespeare))
print()
print(texto_shakespeare[:200])

Cantidad total de caracteres: 1115394

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## 2. Construyendo el vocabulario de caracteres

Las redes neuronales trabajan con numeros, no con letras. Como vamos a trabajar a nivel de caracter, tenemos que asignarle un numero entero (un `id`) a cada caracter distinto que aparece en el texto. Ese conjunto de caracteres distintos es nuestro *vocabulario*.

Convertimos todo a minuscula primero, para no duplicar caracteres por mayusculas/minusculas y mantener el vocabulario mas chico.

In [3]:
vocabulario = sorted(set(texto_shakespeare.lower())) # set() nos proporciona los caracteres únicos que aparecen en el texto.
tam_vocabulario = len(vocabulario)

print("Cantidad de caracteres distintos:", tam_vocabulario)
print("Vocabulario:", "".join(vocabulario))

Cantidad de caracteres distintos: 39
Vocabulario: 
 !$&',-.3:;?abcdefghijklmnopqrstuvwxyz


In [4]:
char_to_id = {caracter: indice for indice, caracter in enumerate(vocabulario)}  # Creamos un diccionario que asigna un ID numérico único a cada carácter

id_to_char = {indice: caracter for indice, caracter in enumerate(vocabulario)}  # Creamos el diccionario inverso para convertir cada ID nuevamente en su carácter


def codificar_texto(texto):
    return np.array([char_to_id[caracter] for caracter in texto.lower()])  # Convertimos cada carácter del texto en su ID numérico


def decodificar_texto(ids_caracteres):
    return "".join([id_to_char[id_caracter] for id_caracter in ids_caracteres])  # Convertimos los IDs nuevamente en caracteres y los unimos en un texto


# Probemos las dos funciones
codificado = codificar_texto("Hello, world!")
print("Codificado:", codificado)
print("Decodificado:", decodificar_texto(codificado))

Codificado: [20 17 24 24 27  6  1 35 27 30 24 16  2]
Decodificado: hello, world!


## 3. Preparando el dataset con ventanas

Ahora mismo tenemos una unica secuencia gigante de caracteres. Para entrenar una RNN, la convertimos en muchas ventanas cortas de largo fijo: cada muestra de entrada es una ventana de texto, y su etiqueta es esa misma ventana pero desplazada un caracter hacia adelante (el modelo tiene que aprender a predecir "el proximo caracter" en cada posicion).

Por ejemplo, con una ventana de largo 10, si el texto original es `"to be or not to be"`, una muestra podria ser:

- entrada: `"to be or n"`
- etiqueta: `"o be or no"` (el mismo texto, corrido un caracter)

En Keras, esto se arma facilmente con `tf.data.Dataset`, usando `window()` para cortar la secuencia en ventanas solapadas.

Vamos a crear un funcion que prepara los datos para este proceso:  


```
Texto completo
      ↓
tf.data.Dataset
      ↓
window()
      ↓
Ventanas solapadas
      ↓
┌─────────────────┐
│ Entrada         │ → "to be or n"
│ Etiqueta        │ → "o be or no"
└─────────────────┘
      ↓
     RNN
```   
Las windows funciona asi:
```
Texto original:
to be or not to be
^^^^^^^^^^
 window 1

to be or not to be
 ^^^^^^^^^^
 window 2

to be or not to be
  ^^^^^^^^^^
 window 3
```



In [5]:
def crear_dataset(texto_codificado, largo_ventana, tam_lote, mezclar=False, seed=None):
    # texto_codificado: texto convertido de caracteres a IDs numéricos
    # largo_ventana: cantidad de caracteres que tendrá cada secuencia
    # tam_lote: cantidad de secuencias que procesamos juntas en cada batch
    # mezclar: indica si queremos mezclar las secuencias antes de crear los batches
    # seed: permite obtener siempre la misma mezcla cuando usamos shuffle

    ds = tf.data.Dataset.from_tensor_slices(texto_codificado) # Convertimos el texto codificado en un Dataset de TensorFlow (a format that TensorFlow can process)

    ds = ds.window(
        largo_ventana + 1, # extra character needed to create the "next character" target
        shift=1,
        drop_remainder=True
    )  # Creamos ventanas consecutivas de largo_ventana + 1 caracteres

    ds = ds.flat_map(
        lambda ventana: ventana.batch(largo_ventana + 1)
    )  # Convertimos cada ventana en un tensor con todos sus caracteres

    if mezclar:
        ds = ds.shuffle(
            100_000,
            seed=seed
        )  # Mezclamos las secuencias para que el modelo no siempre las vea en el mismo orden

    ds = ds.batch(tam_lote)
    # Agrupamos varias secuencias en un batch (Por ejemplo, tam_lote=64 significa 64 secuencias por batch)

    ds = ds.map(
        lambda ventana: (ventana[:, :-1], ventana[:, 1:])
    )  # Separamos cada secuencia en entrada y etiqueta desplazando un carácter

    return ds.prefetch(1) # Preparamos el siguiente batch mientras el modelo procesa el actual

### Armando los splits

El texto completo tiene mas de un millon de caracteres. Para que el entrenamiento sea rapido en clase, vamos a usar solo una porcion (podes agrandarla si queres mejores resultados).

In [6]:
texto_codificado = codificar_texto(texto_shakespeare)

largo_ventana = 50
tam_lote = 64

# Usamos una porcion del texto para que el entrenamiento sea rapido en clase

# Creamos el train set usando los primeros 200,000 caracteres y mezclamos las muestras
set_entrenamiento = crear_dataset(texto_codificado[:200_000], largo_ventana, tam_lote, mezclar=True, seed=42)

# Creamos el validation set usando los siguientes 20,000 caracteres
set_validacion = crear_dataset(texto_codificado[200_000:220_000], largo_ventana, tam_lote)

# Creamos el test set usando los siguientes 20,000 caracteres
set_prueba = crear_dataset(texto_codificado[220_000:240_000], largo_ventana, tam_lote)



```
set_entrenamiento:

┌─────────────────────────────────────┐
│              Batch 1                │
│                                     │
│  ┌───────────────────────────────┐  │
│  │ Secuencia 1                   │  │
│  │ Input  → 50 caracteres        │  │
│  │ Target → 50 caracteres        │  │
│  └───────────────────────────────┘  │
│                                     │
│  ┌───────────────────────────────┐  │
│  │ Secuencia 2                   │  │
│  │ Input  → 50 caracteres        │  │
│  │ Target → 50 caracteres        │  │
│  └───────────────────────────────┘  │
│                                     │
│  ┌───────────────────────────────┐  │
│  │ Secuencia 3                   │  │
│  │ Input  → 50 caracteres        │  │
│  │ Target → 50 caracteres        │  │
│  └───────────────────────────────┘  │
│                                     │
│  │ ... 64 secuencias en total ...│  │
└─────────────────────────────────────┘

┌─────────────────────────────────────┐
│              Batch 2                │
│          64 secuencias              │  
└─────────────────────────────────────┘

┌─────────────────────────────────────┐
│              Batch 3                │
│          64 secuencias              │  
└─────────────────────────────────────┘

                 ...
```



In [7]:
## Veamos los que contiene un sets, usando "take(1)"

for entrada, etiqueta in set_entrenamiento.take(1):  # Tomamos un solo batch del train set (take(1) --> "Give me only the first batch from set_entrenamiento." )

    entrada_texto = decodificar_texto(entrada[0].numpy())  # Convertimos la primera entrada de IDs a texto
    etiqueta_texto = decodificar_texto(etiqueta[0].numpy())  # Convertimos la primera etiqueta de IDs a texto

    print("Entrada:")
    print(entrada_texto)

    print("\nEtiqueta:")
    print(etiqueta_texto)

Entrada:
nd fight with t'other,
ere stay behind this busine

Etiqueta:
d fight with t'other,
ere stay behind this busines


## 4. Por que no usar los IDs directamente? <br> El concepto de **Embedding**

Los IDs de caracter son solo etiquetas arbitrarias (que el caracter "a" sea el id 13 y la "b" el id 14 no significa que se parezcan). Por eso, el primer paso del modelo va a ser una capa `Embedding`, que aprende una representacion densa de cada caracter durante el entrenamiento, en vez de usar el id crudo.

**¿Qué es un Embedding?**

Hasta ahora, cada carácter está representado por un **ID**:

```text
"a" → 13
"b" → 14
"c" → 15
...
"z" → 38
```

Que "a" tenga el ID 13 y "b" tenga el ID 14 no significa que sean parecidos.
<br><br>


--> Usamos una capa Embedding para convertir cada ID en un vector de números:


```
┌─────────────┐
│  Carácter   │
│     "a"     │
└──────┬──────┘
       ↓
┌─────────────┐
│     ID      │
│     13      │
└──────┬──────┘
       ↓
┌─────────────────────────┐
│       Embedding         │
│                         │
│ [0.2, 0.8, -0.1, 0.4]   │
└───────────┬─────────────┘
            ↓
       ┌─────────┐
       │   RNN   │
       └─────────┘
```
El vector podría tener, por ejemplo, 4 valores:  
```
"a" → [ 0.2,  0.8, -0.1,  0.4]
"b" → [ 0.3,  0.7, -0.2,  0.5]
"z" → [-0.8,  0.1,  0.9, -0.4]
```

**--> Los Embeddings se aprende durante el entrenamiento**

Al principio, el Embedding tiene valores iniciales aleatoria. Durante el entrenamiento, el modelo los va ajustando para encontrar una representación útil para la tarea de predecir el próximo carácter.


## 5. Construyendo el modelo

Vamos a usar:

1. Una capa `Embedding`, para convertir cada id de caracter en un vector denso (vector denso = casi no hay zeros).
2. Dos capas `GRU` apiladas (una variante de RNN, similar a LSTM pero un poco mas simple), que van a ir acumulando memoria a lo largo de la secuencia.
3. Una capa `Dense` de salida con `tam_vocabulario` neuronas y activacion `softmax`, para predecir una probabilidad por cada caracter posible.

Una diferencia con lo que vimos en Fashion MNIST: aca queremos una prediccion en **cada paso de tiempo**, no solo al final de la secuencia. Por eso usamos `return_sequences=True` en las capas `GRU`: en vez de devolver un unico vector resumen al final, devuelven un vector por cada caracter de la ventana. La capa `Dense` final se aplica automaticamente a cada uno de esos pasos.

In [8]:
modelo = keras.Sequential([
    layers.Input(shape=(largo_ventana,)), # Definimos que cada entrada contiene una secuencia de 50 IDs
    layers.Embedding(input_dim=tam_vocabulario, output_dim=16), # Numero total de caracteres posibles, cada caracter se convierte en un vector de 16 valores
    layers.GRU(128, return_sequences=True), # La GRU devuelve una salida para cada caracter de la secuencia
    layers.GRU(128, return_sequences=True), # La segunda GRU tambien devuelve una salida para cada caracter
    layers.Dense(tam_vocabulario, activation="softmax"), # Produce una probabilidad para cada caracter posible
])

modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 16)         │           624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 50, 128)        │        56,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 50, 128)        │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 50, 39)         │         5,031 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160,791 (628.09 KB)

 Trainable params: 160,791 (628.09 KB)

 Non-trainable params: 0 (0.00 B)

### Compilando y entrenando

Misma configuracion de siempre: `adam`, `sparse_categorical_crossentropy` (porque las etiquetas son ids enteros), y `accuracy` como metrica.

Entrenar un char-RNN sobre texto real toma bastante mas tiempo que Fashion MNIST. Si tenes poco tiempo, reduci la cantidad de epocas.

In [9]:
modelo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

historial = modelo.fit(
    set_entrenamiento,
    validation_data=set_validacion, # decimos que ya tenemos un validation set
    epochs=5,
)

Epoch 1/5
   3124/Unknown 64s 15ms/step - accuracy: 0.4311 - loss: 1.9538

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


3125/3125 ━━━━━━━━━━━━━━━━━━━━ 69s 16ms/step - accuracy: 0.5191 - loss: 1.5986 - val_accuracy: 0.5206 - val_loss: 1.6160
Epoch 2/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 67s 17ms/step - accuracy: 0.6239 - loss: 1.2045 - val_accuracy: 0.5115 - val_loss: 1.7233
Epoch 3/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 77s 20ms/step - accuracy: 0.6663 - loss: 1.0686 - val_accuracy: 0.5009 - val_loss: 1.8652
Epoch 4/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 62s 16ms/step - accuracy: 0.6934 - loss: 0.9863 - val_accuracy: 0.4923 - val_loss: 1.9766
Epoch 5/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 57s 14ms/step - accuracy: 0.7111 - loss: 0.9343 - val_accuracy: 0.4866 - val_loss: 2.0657


## 6. Prediciendo el proximo caracter

Probemos el modelo con el clasico ejemplo "to be or not to b", que deberia continuar con la letra "e".

In [21]:
texto_prueba = "to be or not to b"
texto_prueba_codificado = codificar_texto(texto_prueba)[np.newaxis, :]  # agregamos la dimension del batch (con '[np.newaxis, :] ')

probabilidades = modelo.predict(texto_prueba_codificado, verbose=0)[0, -1]  # solo nos interesa el ultimo paso de tiempo
id_predicho = np.argmax(probabilidades)

print("Caracter predicho:", id_to_char[id_predicho])

Caracter predicho: e


## 7. Generando texto nuevo

Una vez que la RNN está entrenada, podemos usarla para **generar texto carácter por carácter**.

La idea es sencilla:

```text
Texto inicial
      ↓
     RNN
      ↓
Probabilidades del próximo carácter
      ↓
Elegimos un carácter
      ↓
Lo agregamos al texto
      ↓
Volvemos a pasar el texto por la RNN
      ↓
Probabilidades del siguiente carácter
      ↓
     ...
```

**¿Cómo elegimos el próximo carácter?**
El modelo no nos da directamente un carácter. Nos da una probabilidad para cada carácter posible.  

Por ejemplo:
```
┌─────────────────────────┐
│ Próximo carácter        │
├─────────────────────────┤
│ "e" → 70%               │
│ "a" → 15%               │
│ "m" → 10%               │
│ "t" →  5%               │
└─────────────────────────┘
```
En lugar de elegir siempre el carácter con la probabilidad más alta ("e"), podemos samplear usando estas probabilidades. Esto significa que "e" será elegida aproximadamente el 70% de las veces, pero no siempre.

**El concepto de Temperatura**  
Vamos a agregar un parametro que se llama 'temperatura', que nos permite controlar cuanto nos alejamos de las predicciones mas probables: temperatura baja produce texto mas conservador (y repetitivo); temperatura alta produce texto mas variado (y mas caotico).
<br><br>
La temperatura modifica la distribución de probabilidades antes de hacer el sampling.

Con una temperatura baja, la "e" tendrá todavía más peso:

```
e → █████████████████
a → ██
m → █
t → ▏
```


Con una temperatura alta, las probabilidades se vuelven más parecidas:

```
e → ███████
a → █████
m → ████
t → ███
```

Temperatura baja → más conservador, predecible y repetitivo.  
Temperatura alta → más variado, impredecible y potencialmente caótico.

In [34]:
def proximo_caracter(modelo, texto, temperatura=1): # temperatura controla que tan conservador o variado sera el sampling
    texto_codificado = codificar_texto(texto)[np.newaxis, :] # Convertimos el texto en IDs y agregamos la dimension del batch (con '[np.newaxis, :]')
    y_probas = modelo.predict(texto_codificado, verbose=0)[0, -1:, :]  # Obtenemos las probabilidades del ultimo caracter de la secuencia
    logits = np.log(y_probas) / temperatura  # Ajustamos las probabilidades segun la temperatura
    id_predicho = tf.random.categorical(logits, num_samples=1)[0, 0].numpy()  # Sampleamos un caracter usando las probabilidades en vez de elegir siempre el maximo
    return id_to_char[id_predicho] # Convertimos el ID elegido nuevamente en un caracter


def extender_texto(modelo, texto, n_caracteres=80, temperatura=1):
    for _ in range(n_caracteres): # Repetimos el proceso para generar nuevos caracteres
        texto += proximo_caracter(modelo, texto, temperatura) # Generamos un caracter y lo agregamos al final del texto
    return texto

Probemos con temperaturas baja, media y alta:

In [35]:
print("--- Temperatura baja (0.2) ---")
print(extender_texto(modelo, "to be or not to b", temperatura=0.2))
print()
print("--- Temperatura media (1.0) ---")
print(extender_texto(modelo, "to be or not to b", temperatura=1.0))
print()
print("--- Temperatura alta (2.0) ---")
print(extender_texto(modelo, "to be or not to b", temperatura=2.0))

--- Temperatura baja (0.2) ---
to be or not to be a rorn of him; for what is this true but his trades to the people,
being no re

--- Temperatura media (1.0) ---
to be or not to be
pownt o'er trust of sews of worshiol
shall to passing here.

aufidius:
mar unn

--- Temperatura alta (2.0) ---
to be or not to braw;
ols; then neithorteny, lord, peoes 'ges
which sarriase pass certain;
were b


Con temperatura baja, es comun ver que el modelo repite las mismas palabras o frases. Con temperatura media, el texto suele ser mas convincente (aunque siga sin tener mucho sentido). Con temperatura muy alta, el texto se vuelve casi aleatorio.

Con pocas epocas de entrenamiento y una porcion chica del texto, no esperes resultados tan prolijos como los que se logran entrenando sobre el texto completo durante mas tiempo -- pero el mecanismo es exactamente el mismo.

In [38]:
print(extender_texto(modelo, "Hi, my name is ", temperatura=0.2))

Hi, my name is gone, intended my heart
as thou hast to deny him he leaves
are straight the peop


In [33]:
texto_prueba = "Hi, my name is "

for _ in range(5):  # Repetimos el proceso 5 veces para generar 5 caracteres
    texto_prueba_codificado = codificar_texto(texto_prueba)[np.newaxis, :]  # Codificamos el texto y agregamos la dimension del batch
    probabilidades = modelo.predict(texto_prueba_codificado,verbose=0)[0, -1]  # Obtenemos las probabilidades del ultimo paso de tiempo

    id_predicho = np.argmax(probabilidades)  # Elegimos el caracter con mayor probabilidad
    caracter_predicho = id_to_char[id_predicho]  # Convertimos el ID nuevamente en caracter
    texto_prueba += caracter_predicho  # Agregamos el caracter predicho al texto

print("Texto generado:", texto_prueba)

Texto generado: Hi, my name is gone.


## Limitaciones y proximo paso

Este modelo no puede aprender patrones mas largos que `largo_ventana` (50 caracteres en nuestro caso). Aumentar ese numero ayuda, pero como vimos en el notebook anterior, ni siquiera una GRU o una LSTM pueden manejar bien secuencias extremadamente largas: la memoria se va perdiendo. Y sea cual sea el largo, el modelo sigue leyendo el texto de a un caracter por vez, en orden, lo cual es lento.

Esa es exactamente la limitacion que llevo a la creacion de los Transformers, que ya viste conceptualmente: en vez de depender de una memoria secuencial, usan atencion para mirar toda la secuencia a la vez.